In [1]:
import sys
from pathlib import Path

# Define the path to the directory containing compute_rms.py
path_to_compute_rms = Path("../errpca_pt + compute_rms")

# Add this path to sys.path
sys.path.append(str(path_to_compute_rms))


In [37]:
import numpy as np
from numpy.linalg import slogdet
from scipy.sparse import issparse, csr_matrix
from compute_rms import compute_rms  # Ensure this module is available

def cf_full(X, A, S, Mu, V, Av=None, Sv=None, Isv=None, Muv=None, Va=None, Vmu=None, M=None, sXv=None, ndata=None):
    n1, n2 = X.shape
    ncomp = A.shape[1]

    # Handle M, sXv, ndata if not provided
    if M is None:
        M = ~np.isnan(X)
        X = np.where(M, X, 0)

    if sXv is None:
        IX, JX = np.where(M)
        ndata = len(IX)

        # Call compute_rms and capture results
        rms, errMx = compute_rms(X, A, S, M, ndata)

        # Initialize sXv based on rms
        sXv = (rms ** 2) * ndata
        if Sv is None:
            for r in range(ndata):
                a_IXr = A[IX[r], :]
                sv_JXr = np.eye(ncomp)
                sXv += a_IXr @ sv_JXr @ a_IXr.T
                if Av is not None:
                    sXv += S[:, JX[r]].T @ Av[IX[r]] @ S[:, JX[r]] + np.sum(Sv[JX[r]] * Av[IX[r]])
        else:
            for r in range(ndata):
                a_IXr = A[IX[r], :]
                sv_IJXr = Sv[Isv[JX[r]]] if Isv else Sv[JX[r]]
                sXv += a_IXr @ sv_IJXr @ a_IXr.T
                if Av is not None:
                    sXv += S[:, JX[r]].T @ Av[IX[r]] @ S[:, JX[r]] + np.sum(Sv[JX[r]] * Av[IX[r]])

        if Muv:
            sXv += np.sum(Muv[IX])

    # Determine whether to use priors based on Va
    use_prior = Va is not None and not np.any(np.isinf(Va))

    cost_x = 0.5 / V * sXv + 0.5 * ndata * np.log(2 * np.pi * V)
    cost_mu = 0.0
    cost_a = 0.0

    if use_prior:
        if Muv:
            cost_mu = 0.5 / Vmu * np.sum(Mu ** 2 + Muv) - 0.5 * np.sum(np.log(Muv)) + (n1 / 2) * np.log(Vmu) - (n1 / 2)
        elif Vmu != 0:
            cost_mu = 0.5 / Vmu * np.sum(Mu ** 2) + (n1 / 2) * np.log(2 * np.pi * Vmu)

        if Av:
            cost_a = 0.5*np.sum(np.sum(A @ A, axis = 0) / Va) + (n1 / 2) * np.sum(np.log(Va), axis = 0) - (n1 * ncomp / 2)
            print("cost_a after initial assignment:", cost_a)
            for i in range(n1):
                trace_term = 0.5 * np.sum(np.diag(Av[i]) / Va)
                sign, logdet = slogdet(Av[i])
                cost_a += trace_term - 0.5 * logdet if sign > 0 else trace_term - 0.5 * (-np.inf)
                print(f"cost_a after updating with trace and determinant of Av[{i}]:", cost_a)
        else:
            cost_a = 0.5 * np.sum(A ** 2 / Va) + (n1 / 2) * np.sum(np.log(2 * np.pi * Va))
            print("cost_a after assignment with Va:", cost_a)
    else:
        if Muv:
            cost_mu = -0.5 * np.sum(np.log(2 * np.pi * Muv)) - (n1 / 2)
        if Av:
            cost_a = - (n1 * ncomp / 2) * (1 + np.log(2 * np.pi))
            print("cost_a after no-prior initial assignment:", cost_a)
            for i in range(n1):
                sign, logdet = slogdet(Av[i])
                cost_a -= 0.5 * logdet if sign > 0 else 0.5 * (-np.inf)
                print(f"cost_a after updating with determinant of Av[{i}]:", cost_a)

    cost_s = 0.5 * np.sum(S ** 2)
    if Sv:
        if Isv:
            for j in range(n2):
                sv_idx = Isv[j]
                trace_svj = 0.5 * np.trace(Sv[sv_idx])
                sign, logdet_svj = slogdet(Sv[sv_idx])
                cost_s += trace_svj - 0.5 * logdet_svj if sign > 0 else trace_svj - 0.5 * (-np.inf)
        else:
            for j in range(n2):
                trace_svj = 0.5 * np.trace(Sv[j])
                sign, logdet_svj = slogdet(Sv[j])
                cost_s += trace_svj - 0.5 * logdet_svj if sign > 0 else trace_svj - 0.5 * (-np.inf)
    cost_s -= (ncomp * n2) / 2
    cost = cost_mu + cost_a + cost_x + cost_s


    return cost, cost_x, cost_a, cost_mu, cost_s


In [17]:
# Test Case 1: Basic Functionality
import numpy as np
# from cf_full import cf_full

# Define inputs
X = np.array([[1, 2],
              [3, 4]])
A = np.array([[1, 0],
              [0, 1]])
S = np.array([[1, 1],
              [1, 1]])
Mu = np.array([0.5, 0.5])
V = 1
Av = [np.eye(2), np.eye(2)]
Sv = [np.eye(2), np.eye(2)]
Isv = None
Muv = None
Va = np.array([1, 1])
Vmu = 1
M = ~np.isnan(X)
sXv = None
ndata = None

# Compute cost
cost, cost_x, cost_a, cost_mu, cost_s = cf_full(X, A, S, Mu, V, Av, Sv, Isv, Muv, Va, Vmu, M, sXv, ndata)

# Display results
print('Test Case 1 Results:')
print(f'cost: {cost}')
print(f'cost_x: {cost_x}')
print(f'cost_a: {cost_a}')
print(f'cost_mu: {cost_mu}')
print(f'cost_s: {cost_s}')


befor is sparse
cost_a after initial assignment: -1.0
cost_a after updating with trace and determinant of Av[0]: 0.0
cost_a after updating with trace and determinant of Av[1]: 1.0
Computed costs:
Total Cost: 25.763631199228033
Cost_x: 20.67575413281869
Cost_a: 1.0
Cost_mu: 2.0878770664093453
Cost_s: 2.0
Test Case 1 Results:
cost: 25.763631199228033
cost_x: 20.67575413281869
cost_a: 1.0
cost_mu: 2.0878770664093453
cost_s: 2.0


In [27]:
# Test Case 2: Handling Missing Values
import numpy as np
# from cf_full import cf_full

# Define inputs
X = np.array([[1, np.nan],
              [3, 4]])
A = np.array([[1, 0],
              [0, 1]])
S = np.array([[1, 1],
              [1, 1]])
Mu = np.array([0.5, 0.5])
V = 1
Av = [np.eye(2), np.eye(2)]  # Correct: list of NumPy arrays
Sv = [np.eye(2), np.eye(2)]  # Correct: list of NumPy arrays
Isv = []
Muv = []
Va = np.array([1, 1])
Vmu = 1
M = ~np.isnan(X)
sXv = None
ndata = None

# Compute cost
cost, cost_x, cost_a, cost_mu, cost_s = cf_full(
    X, A, S, Mu, V, Av, Sv, Isv, Muv, Va, Vmu, M, sXv, ndata
)

# Display results
print('Test Case 2 Results:')
print(f'cost: {cost}')
print(f'cost_x: {cost_x}')
print(f'cost_a: {cost_a}')
print(f'cost_mu: {cost_mu}')
print(f'cost_s: {cost_s}')


befor is sparse
cost_a after initial assignment: -1.0
cost_a after updating with trace and determinant of Av[0]: 0.0
cost_a after updating with trace and determinant of Av[1]: 1.0
Computed costs:
Total Cost: nan
Cost_x: nan
Cost_a: 1.0
Cost_mu: 2.0878770664093453
Cost_s: 2.0
Test Case 2 Results:
cost: nan
cost_x: nan
cost_a: 1.0
cost_mu: 2.0878770664093453
cost_s: 2.0


In [39]:
import numpy as np
from scipy.sparse import csr_matrix
from compute_rms import compute_rms  # Assuming compute_rms is defined
# from cf_full import cf_full  # Assuming cf_full is defined

# Define the sparse matrix X_sparse directly in Python
X_sparse = csr_matrix([[1, 0], [0, 4]])  # Equivalent to MATLAB's sparse([1, 0; 0, 4])

# Define fixed values for A and S to match MATLAB test
A = np.array([[0.49671415, -0.1382643], 
              [0.64768854, 1.52302986]])  # Fixed matrix A (2x2)
S = np.array([[-0.23415337, -0.23413696], 
              [1.57921282, 0.76743473]])  # Fixed matrix S (2x2)
Mu = np.zeros((2, 1))  # Mean vector, 2x1
V = 1  # Scalar variance
Av = [np.eye(2), np.eye(2)]  # List of 2 identity matrices, equivalent to MATLAB's cell array
Sv = [np.eye(2), np.eye(2)]  # List of 2 identity matrices
Isv = []  # Empty list, equivalent to MATLAB empty array
Muv = []  # Empty list
Va = np.array([1, 1])  # Prior variances, 2x1
Vmu = 1  # Scalar prior variance
M = ~np.isnan(X_sparse.toarray())  # Mask matrix, 2x2, similar to MATLAB's M
sXv = None  # Empty, equivalent to MATLAB empty array
ndata = 2  # Set ndata to 2 to match MATLAB test

# Call cf_full function with the same variables as MATLAB
cost, cost_x, cost_a, cost_mu, cost_s = cf_full(X_sparse, A, S, Mu, V, Av, Sv, Isv, Muv, Va, Vmu, M, sXv, ndata)

# Display results for comparison with MATLAB
print("\nTest Case 3 Results (Python):")
print("Total Cost:", cost)
print("Cost_x:", cost_x)
print("Cost_a:", cost_a)
print("Cost_mu:", cost_mu)
print("Cost_s:", cost_s)


befor is sparse
inside is sparse
Arguments sent to errpca_pt:
X_data: [1. 4.]
X_indices: [0 1]
X_indptr: [0 1 2]
A: [[ 0.49671415 -0.1382643 ]
 [ 0.64768854  1.52302986]]
S: [[-0.23415337 -0.23413696]
 [ 1.57921282  0.76743473]]
numCPU: 1
Returned from errpca_pt:
errMx data: [1.33465605 2.98282182]
errMx indices: [0 1]
errMx indptr: [0 1 2]
errMx shape: (2, 2)
cost_a after initial assignment: -0.2919264733257996 2 2
cost_a after updating with trace and determinant of Av[0]: 0.7080735266742004
cost_a after updating with trace and determinant of Av[1]: 1.7080735266742004

Test Case 3 Results (Python):
Total Cost: 24.354709133651053
Cost_x: 19.212499984376045
Cost_a: 1.7080735266742004
Cost_mu: 1.8378770664093453
Cost_s: 1.5962585561914615
